In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 170)
np.set_printoptions(suppress=True)

In [3]:
import os
from google.colab import files
files.upload()

DATA, BOOK = 'knhanes_like_week4.csv', 'codebook_week4.csv'
df = pd.read_csv(DATA)
w  = df['wt'].to_numpy()
ST, PS = df['kstrata'].to_numpy(), df['psu'].to_numpy()

print(f'n = {len(df):,}  |  조사구 {df.psu.nunique()}개  |  층 {df.kstrata.nunique()}개')
print(f'가중치 합 = {w.sum():,.0f}')

Saving codebook_week4.csv to codebook_week4.csv
Saving knhanes_like_week4.csv to knhanes_like_week4.csv
n = 5,367  |  조사구 263개  |  층 18개
가중치 합 = 450,899


In [4]:
import numpy as np
import pandas as pd

# 1. 데이터 불러오기
df = pd.read_csv('knhanes_like_week4.csv')

# 2. 전처리 (무응답 코드 9 처리 및 노출/비노출 정의)
# edu: 1, 2 -> 저학력(노출군), 3, 4 -> 고졸 이상(비노출군), 9/NaN -> 결측 제외
df['exposure'] = np.nan
df.loc[df['edu'].isin([1, 2]), 'exposure'] = 1
df.loc[df['edu'].isin([3, 4]), 'exposure'] = 0

# income: 무응답(9)을 NaN 처리 및 소득 1분위(범주형 0/1) 변수 생성
df['income_clean'] = df['income'].replace(9, np.nan)
df['income_q1'] = np.where(df['income_clean'].isna(), np.nan, (df['income_clean'] == 1).astype(float))

# 범주형 분석 변수 정제 (0/1 binary indicator로 변환)
df['female'] = np.where(df['sex'].isna(), np.nan, (df['sex'] == 2).astype(float))
df['eupmyeon'] = np.where(df['urban'].isna(), np.nan, (df['urban'] == 2).astype(float))
df['smk_curr'] = np.where(df['smk'].isna(), np.nan, (df['smk'] == 1).astype(float))
df['drink_high'] = np.where(df['drink'].isna(), np.nan, (df['drink'] == 3).astype(float))
df['exercise_yes'] = np.where(df['exercise'].isna(), np.nan, (df['exercise'] == 2).astype(float))
df['htn_yes'] = np.where(df['htn'].isna(), np.nan, (df['htn'] == 1).astype(float))


# 3. 복합표본 도메인 분석 함수 (Taylor Series Linearization 기반 Design-based SE)
def svy_domain_analysis(data, var_name, is_categorical=False):
    """
    Subpopulation(Domain)별 가중 추정치, 설계 기반 SE 및 비가중 통계량을 계산
    """
    # 관련 변수 결측 제외
    sub_df = data.dropna(subset=[var_name, 'exposure', 'wt', 'kstrata', 'psu']).copy()

    # 범주형 변수의 경우 % 단위 변환을 위해 100을 곱함
    if is_categorical:
        sub_df['y'] = sub_df[var_name] * 100.0
    else:
        sub_df['y'] = sub_df[var_name]

    # --- [전체 모집단 가중 추정치 & SE] ---
    w_tot = sub_df['wt'].sum()
    mean_tot = (sub_df['y'] * sub_df['wt']).sum() / w_tot

    # Linearized variable for total
    sub_df['u_tot'] = (sub_df['wt'] / w_tot) * (sub_df['y'] - mean_tot)

    # Strata & PSU variance calculation
    psu_tot = sub_df.groupby(['kstrata', 'psu'])['u_tot'].sum().reset_index()
    st_means_tot = psu_tot.groupby('kstrata')['u_tot'].mean().reset_index().rename(columns={'u_tot': 'st_mean'})
    psu_tot = psu_tot.merge(st_means_tot, on='kstrata')

    var_tot = 0
    for st, group in psu_tot.groupby('kstrata'):
        nh = len(group)
        if nh > 1:
            var_tot += (nh / (nh - 1)) * np.sum((group['u_tot'] - group['st_mean']) ** 2)
    se_tot = np.sqrt(var_tot)
    unw_mean_tot = sub_df['y'].mean()

    # --- [도메인별(노출군/비노출군) 가중 추정치 & SE] ---
    domain_res = {}
    for d_val in [1, 0]: # 1: 노출군(저학력), 0: 비노출군(고졸 이상)
        d_mask = (sub_df['exposure'] == d_val)
        w_d = sub_df.loc[d_mask, 'wt'].sum()
        mean_d = (sub_df.loc[d_mask, 'y'] * sub_df.loc[d_mask, 'wt']).sum() / w_d

        # Subpopulation linearization (Domain 외부 데이터는 u=0)
        sub_df['u_d'] = 0.0
        sub_df.loc[d_mask, 'u_d'] = (sub_df.loc[d_mask, 'wt'] / w_d) * (sub_df.loc[d_mask, 'y'] - mean_d)

        psu_d = sub_df.groupby(['kstrata', 'psu'])['u_d'].sum().reset_index()
        st_means_d = psu_d.groupby('kstrata')['u_d'].mean().reset_index().rename(columns={'u_d': 'st_mean'})
        psu_d = psu_d.merge(st_means_d, on='kstrata')

        var_d = 0
        for st, group in psu_d.groupby('kstrata'):
            nh = len(group)
            if nh > 1:
                var_d += (nh / (nh - 1)) * np.sum((group['u_d'] - group['st_mean']) ** 2)
        se_d = np.sqrt(var_d)

        unw_n_d = d_mask.sum()
        unw_mean_d = sub_df.loc[d_mask, 'y'].mean()

        domain_res[d_val] = {
            'n': unw_n_d,
            'unw_mean': unw_mean_d,
            'w_mean': mean_d,
            'se': se_d
        }

    return {
        'total': {'n': len(sub_df), 'unw_mean': unw_mean_tot, 'w_mean': mean_tot, 'se': se_tot},
        'exp_1': domain_res[1],
        'exp_0': domain_res[0]
    }


# 4. Table 1 생성 변수 리스트 설정
var_list = [
    ('age', '연령 (세)', False),
    ('female', '여성 (%)', True),
    ('eupmyeon', '읍면 거주 (%)', True),
    ('income_q1', '소득 1분위 (%)', True),
    ('bmi', '체질량지수 (kg/m²)', False),
    ('sbp', '수축기 혈압 (mmHg)', False),
    ('dbp', '이완기 혈압 (mmHg)', False),
    ('chol', '총콜레스테롤 (mg/dL)', False),
    ('glucose', '공복혈당 (mg/dL)', False),
    ('smk_curr', '현재 흡연 (%)', True),
    ('drink_high', '고위험 음주 (%)', True),
    ('exercise_yes', '규칙적 운동 (%)', True),
    ('htn_yes', '고혈압 유병 (%)', True)
]

# 5. 테이블용 데이터 추출 및 DataFrame 구축
rows_table1 = []
rows_b3_comp = []

for var_col, var_label, is_cat in var_list:
    res = svy_domain_analysis(df, var_col, is_categorical=is_cat)

    # Table 1 용 (가중 추정치 + SE 표기)
    fmt_tot = f"{res['total']['w_mean']:.1f} ({res['total']['se']:.1f})"
    fmt_e1  = f"{res['exp_1']['w_mean']:.1f} ({res['exp_1']['se']:.1f})"
    fmt_e0  = f"{res['exp_0']['w_mean']:.1f} ({res['exp_0']['se']:.1f})"

    rows_table1.append([var_label, fmt_tot, fmt_e1, fmt_e0])

    # B-3 질문용 (비가중 vs 가중 비교 표기)
    unit = "%" if is_cat else ""
    rows_b3_comp.append([
        var_label,
        f"{res['total']['unw_mean']:.1f}{unit}", f"{res['total']['w_mean']:.1f}{unit}",
        f"{res['exp_1']['unw_mean']:.1f}{unit}", f"{res['exp_1']['w_mean']:.1f}{unit}",
        f"{res['exp_0']['unw_mean']:.1f}{unit}", f"{res['exp_0']['w_mean']:.1f}{unit}"
    ])

# N 수 확인
n_total = len(df.dropna(subset=['exposure']))
n_exp1  = (df['exposure'] == 1).sum()
n_exp0  = (df['exposure'] == 0).sum()

# Table 1 DataFrame
df_table1 = pd.DataFrame(rows_table1, columns=[
    '특성 변수',
    f'전체 (N={n_total:,})',
    f'노출군: 저학력 (n={n_exp1:,})',
    f'비노출군: 고졸 이상 (n={n_exp0:,})'
])

# B-3 비교 DataFrame
df_b3_comp = pd.DataFrame(rows_b3_comp, columns=[
    '변수명', '전체 비가중', '전체 가중',
    '노출군 비가중', '노출군 가중',
    '비노출군 비가중', '비노출군 가중'
])

# 6. 결과 출력
print("=========================================================================")
print(" [Table 1] 연구 대상자의 일반적 및 임상적 특성 (도메인 분석 SE 적용)")
print("=========================================================================")
display(df_table1)

print("\n=========================================================================")
print(" [B-3 분석용] 비가중 vs 가중 추정치 상세 비교표")
print("=========================================================================")
display(df_b3_comp)

 [Table 1] 연구 대상자의 일반적 및 임상적 특성 (도메인 분석 SE 적용)


,특성 변수,"전체 (N=5,287)","노출군: 저학력 (n=1,761)","비노출군: 고졸 이상 (n=3,526)"
0,연령 (세),49.1 (0.2),60.0 (0.4),45.3 (0.2)
1,여성 (%),51.0 (0.7),51.9 (1.3),50.7 (0.9)
2,읍면 거주 (%),23.0 (0.2),35.8 (0.9),18.5 (0.3)
3,소득 1분위 (%),23.9 (0.6),44.7 (1.3),17.1 (0.7)
4,체질량지수 (kg/m²),23.8 (0.1),24.1 (0.1),23.7 (0.1)
5,수축기 혈압 (mmHg),122.8 (0.4),129.0 (0.6),120.6 (0.5)
6,이완기 혈압 (mmHg),73.4 (0.3),75.8 (0.4),72.5 (0.3)
7,총콜레스테롤 (mg/dL),170.4 (0.5),172.8 (0.9),169.5 (0.6)
8,공복혈당 (mg/dL),91.6 (0.3),94.5 (0.5),90.7 (0.3)
9,현재 흡연 (%),24.2 (0.7),24.6 (1.2),24.0 (0.8)



 [B-3 분석용] 비가중 vs 가중 추정치 상세 비교표


,변수명,전체 비가중,전체 가중,노출군 비가중,노출군 가중,비노출군 비가중,비노출군 가중
0,연령 (세),54.0,49.1,65.1,60.0,48.4,45.3
1,여성 (%),49.4%,51.0%,49.9%,51.9%,49.1%,50.7%
2,읍면 거주 (%),40.2%,23.0%,55.0%,35.8%,32.9%,18.5%
3,소득 1분위 (%),29.1%,23.9%,50.1%,44.7%,19.4%,17.1%
4,체질량지수 (kg/m²),24.0,23.8,24.3,24.1,23.8,23.7
5,수축기 혈압 (mmHg),125.7,122.8,132.5,129.0,122.3,120.6
6,이완기 혈압 (mmHg),74.5,73.4,77.2,75.8,73.2,72.5
7,총콜레스테롤 (mg/dL),172.0,170.4,174.8,172.8,170.7,169.5
8,공복혈당 (mg/dL),92.9,91.6,95.9,94.5,91.5,90.7
9,현재 흡연 (%),24.2%,24.2%,24.7%,24.6%,23.9%,24.0%


## 각주 작성

1. n은 관측 수, %와 평균은 가중 추정치라는 것
2. 연속형 변수를 어떤 통계량으로 요약했는지
3. 집단 비교에 사용한 방법 (SMD 기준 포함)
4. 결측이 있었다면 그 처리 방식

> **각주:**

1.표에 제시된 n은 결측치를 제외한 단순 관측 수이며 백분율(%)과 평균은 복합표본 가중치, 층화변수, 조사구가 반영된 가중 추정치임.

2.연속형 변수는 가중 평균과 가중 표준오차(SE)로 요약하였음.

3.노출군(저학력)과 비노출군(고학력) 간의 집단 비교는 복합표본 도메인 분석과 표준화 평균 차이(SMD)를 활용하였으며 SMD 절댓값이 0.1 미만인 경우는 무시할 만한 차이로 판정함.

4.분석 과정에서 결측값이 존재하는 변수는 해당 변수의 분석 시 결측을 제외하여 처리함.